# Day 6: FBref scraper for complementary football data

**Phase 1: Foundations & Infrastructure**

## Objective
FBref scraper for complementary football data

## Task
- Installed parsing dependencies (`lxml`, `html5lib`, `beautifulsoup4`).
- Used verified Wayback Machine snapshots of FBref to ensure URL stability.
- Bypassed FBref's hidden tables by removing HTML comments before parsing.
- Extracted and parsed HTML tables efficiently using `pandas.read_html`.
- Cleaned the data by flattening complex MultiIndex columns into a single level.
- Developed a reusable `fbref_scraper.py` script featuring:
  - Input validation for Top 5 European Leagues.
  - Ethical web scraping practices (`time.sleep` delays) to respect server limits.
  - Automated and dynamic CSV saving to the `data/raw/fbref/` directory.
- Successfully exported clean datasets for Premier League and La Liga.

**Deliverable:** Script src/scrapers/fbref_scraper.py + demo notebook + downloaded data.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



In [ ]:
pip install lxml

In [ ]:
pip install html5lib

In [ ]:
!pip install --upgrade pandas beautifulsoup4 lxml

In [5]:
import requests
import pandas as pd
from io import StringIO

target_url = "https://web.archive.org/web/20240224151703/https://fbref.com/en/comps/9/stats/Premier-League-Stats"

print("Requesting archived data...")
response = requests.get(target_url)

if response.status_code == 200:
    # 1. Remove comments to reveal hidden tables
    uncommented_html = response.text.replace('', '')
    html_string = StringIO(uncommented_html)
    
    # 2. Extract ALL tables without filtering by ID
    tables = pd.read_html(html_string)
    
    # 3. Check how many we caught
    print(f"Success! Found {len(tables)} tables.")
    
    # Let's inspect the first one
    raw_df = tables[0]
    display(raw_df.head())
else:
    print(f"Failed. Status code: {response.status_code}")

Requesting archived data...
Success! Found 2 tables.


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
               Squad               # Pl                Age               Poss   
0            Arsenal                 25               25.5               61.5   
1        Aston Villa                 28               27.6               55.6   
2        Bournemouth                 27               26.4               44.4   
3          Brentford                 28               27.4               44.4   
4           Brighton                 27               26.7               62.3   

  Playing Time                    Performance      ... Per 90 Minutes        \
            MP Starts   Min   90s         Gls Ast  ...            Gls   Ast   
0           28    308  2520  28.0          67  47  ...           2.39  1.68   
1           27    297  2430  27.0          56  41  ...           2.07  1.52   
2           27    297  2430  27.0          37  27  ...           1.37  1.00   
3           28    308  2520  28.0          39  24  ...           1.39  0.86   
4           27    297  2430  27.0          46  35  ...           1.70  1.30   

                                                        
    G+A  G-PK G+A-PK    xG   xAG xG+xAG  npxG npxG+xAG  
0  4.07  2.11   3.79  2.02  1.41   3.43  1.81     3.21  
1  3.59  1.93   3.44  1.86  1.38   3.24  1.75     3.12  
2  2.37  1.33   2.33  1.49  1.07   2.56  1.43     2.51  
3  2.25  1.29   2.14  1.57  1.09   2.66  1.49     2.58  
4  3.00  1.52   2.81  1.67  1.23   2.90  1.53     2.76  

[5 rows x 32 columns]

In [6]:
# Create an empty list to store our new flat column names
flattened_columns = []

for col in raw_df.columns:
    # If the top level is 'Unnamed', we just keep the bottom level (e.g., 'Squad')
    if 'Unnamed' in col[0]:
        flattened_columns.append(col[1])
    else:
        # Otherwise, we combine both levels to keep the context (e.g., 'Performance_Gls')
        flattened_columns.append(f"{col[0]}_{col[1]}")

# Overwrite the old MultiIndex columns with our new flat list
raw_df.columns = flattened_columns

display(raw_df.head())

,Squad,# Pl,Age,Poss,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Per 90 Minutes_xG,Per 90 Minutes_xAG,Per 90 Minutes_xG+xAG,Per 90 Minutes_npxG,Per 90 Minutes_npxG+xAG
0,Arsenal,25,25.5,61.5,28,308,2520,28.0,67,47,...,2.39,1.68,4.07,2.11,3.79,2.02,1.41,3.43,1.81,3.21
1,Aston Villa,28,27.6,55.6,27,297,2430,27.0,56,41,...,2.07,1.52,3.59,1.93,3.44,1.86,1.38,3.24,1.75,3.12
2,Bournemouth,27,26.4,44.4,27,297,2430,27.0,37,27,...,1.37,1.00,2.37,1.33,2.33,1.49,1.07,2.56,1.43,2.51
3,Brentford,28,27.4,44.4,28,308,2520,28.0,39,24,...,1.39,0.86,2.25,1.29,2.14,1.57,1.09,2.66,1.49,2.58
4,Brighton,27,26.7,62.3,27,297,2430,27.0,46,35,...,1.70,1.30,3.00,1.52,2.81,1.67,1.23,2.90,1.53,2.76


In [9]:
raw_df.to_csv("/Users/davidbazalduamendez/Documents/GitHub/100-days-Data-Sports-Challenge/data/raw/fbref/fbref_premier_league_2023.csv", index=False)